<a href="https://colab.research.google.com/github/tanvi-patil1/Real-Time-AI-Captioning-System-using-Speech-and-Facial-Movement-Analysis/blob/main/pbl5thsem.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install torch transformers opencv-python whisper numpy

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.8/42.8 kB 4.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for whisper: filename=whisper-1.1.10-py3-none-any.whl size=41120 sha256=ebdcef451a867aef487c516c2654d325c5d83e8c3060b84912e7e3186e9c9f6e
  Stored in directory: /root/.cache/pip/wheels/34/b8/4e/9c4c3351d670e06746a340fb4b7d854c76517eec225e5b32b1
Successfully built whisper


In [ ]:
pip install openai-whisper

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 30.4 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for openai-whisper: filename=openai_whisper-20250625-py3-none-any.whl size=803979 sha256=4c46f88df5ab28aa73e5f0a839846e478cd85e0bb9971b5c8a11529553a8c38a
  Stored in directory: /root/.cache/pip/wheels/61/d2/20/09ec9bef734d126cba375b15898010b6cc28578d8afdde5869
Successfully built openai-whisper


In [ ]:
!pip install git+https://github.com/openai/whisper.git

  Cloning https://github.com/openai/whisper.git to /tmp/pip-req-build-qgp4jjce
  Running command git clone --filter=blob:none --quiet https://github.com/openai/whisper.git /tmp/pip-req-build-qgp4jjce
  Resolved https://github.com/openai/whisper.git to commit cba3768142a28276a90f14907e4900372c0c3ee0
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [ ]:
import torch

import whisper

import cv2

import numpy as np

from transformers import T5Tokenizer, T5ForConditionalGeneration, BertTokenizer, BertModel

from transformers import pipeline  # For optional sentiment/context

In [ ]:

import gradio as gr

In [ ]:
print("Loading models...")
import whisper
from transformers import T5Tokenizer, T5ForConditionalGeneration, BertTokenizer, BertModel

asr_model = whisper.load_model("base")  # ASR: Whisper (multilingual, noise-robust)
t5_tokenizer = T5Tokenizer.from_pretrained("t5-small")
t5_model = T5ForConditionalGeneration.from_pretrained("t5-small")  # Fusion model
bert_tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
bert_model = BertModel.from_pretrained("bert-base-uncased")  # Context-awareness
print("Models loaded.")

Loading models...


100%|████████████████████████████████████████| 139M/139M [00:01<00:00, 124MiB/s]
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Models loaded.


In [ ]:
import kagglehub
import os

# Download the dataset
path = kagglehub.dataset_download("mohamedbentalb/lipreading-dataset")
print("Path to dataset files:", path)

# Explore the dataset (check folders)
print("Contents of dataset:", os.listdir(path))
# Look for folders like 'videos' and 'transcripts' – note the exact names for Step 2

100%|██████████| 404M/404M [00:10<00:00, 39.8MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/mohamedbentalb/lipreading-dataset/versions/1
Contents of dataset: ['data']


In [ ]:
import os

# List files in the alignments directory
alignments_path = os.path.join(path, 'data', 'alignments')
print("Contents of alignments directory:", os.listdir(alignments_path))

Contents of alignments directory: ['s1']


In [ ]:
# Explore the data directory to find the correct paths
data_path = os.path.join(path, 'data')
print("Contents of data directory:", os.listdir(data_path))

Contents of data directory: ['alignments', 's1']


In [ ]:
import os
data_dir = os.path.join(path, 'data')
print("Contents of 'data' folder:", os.listdir(data_dir))
# This will show subfolders like 'videos', 'transcripts', etc.

Contents of 'data' folder: ['alignments', 's1']


In [ ]:
import os
s1_dir = os.path.join(path, 'data', 's1')
alignments_dir = os.path.join(path, 'data', 'alignments')
print("Contents of 's1':", os.listdir(s1_dir)[:5])  # First 5 files
print("Contents of 'alignments':", os.listdir(alignments_dir)[:5])  # First 5 files

Contents of 's1': ['lbaq7a.mpg', 'prwkzp.mpg', 'bgan4n.mpg', 'bbbz8n.mpg', 'pbao8n.mpg']
Contents of 'alignments': ['s1']


In [ ]:
import cv2
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
import os

# Function to extract mouth from a video frame
def extract_mouth(frame):
    face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    faces = face_cascade.detectMultiScale(gray, 1.1, 4)
    if len(faces) > 0:
        x, y, w, h = faces[0]
        mouth_roi = gray[y + int(h/2):y + h, x:x + w]
        mouth_roi = cv2.resize(mouth_roi, (64, 64))
        return mouth_roi / 255.0
    return np.zeros((64, 64))

# Dataset class to load videos and transcripts
class LipReadingDataset(Dataset):
    def __init__(self, video_dir, transcript_dir, max_frames=25, num_samples=None):
        self.video_dir = video_dir
        self.transcript_dir = transcript_dir
        self.max_frames = max_frames
        self.data = []
        for file in os.listdir(video_dir):
            if file.endswith('.mpg'):  # Videos are .mpg
                video_path = os.path.join(video_dir, file)
                transcript_path = os.path.join(transcript_dir, file.replace('.mpg', '.align'))  # Transcripts are .align
                if os.path.exists(transcript_path):
                    with open(transcript_path, 'r') as f:
                        # GRID .align files have lines like "0 1000 bin" – extract the text part
                        lines = f.readlines()
                        transcript = ' '.join([line.split()[-1] for line in lines if line.strip()]).lower()
                        self.data.append((video_path, transcript))

        if num_samples is not None:
            self.data = self.data[:num_samples]

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        video_path, transcript = self.data[idx]
        cap = cv2.VideoCapture(video_path)
        frames = []
        while len(frames) < self.max_frames:
            ret, frame = cap.read()
            if not ret: break
            mouth = extract_mouth(frame)
            frames.append(mouth)
        cap.release()
        while len(frames) < self.max_frames:
            frames.append(np.zeros((64, 64)))
        frames = np.array(frames[:self.max_frames])
        frames = torch.tensor(frames, dtype=torch.float32).unsqueeze(1)  # Add channel dim: (20, 1, 64, 64)
        vocab = "abcdefghijklmnopqrstuvwxyz "
        transcript_indices = [vocab.index(c) for c in transcript if c in vocab][:10]
        transcript_indices += [len(vocab)] * (10 - len(transcript_indices))
        return frames, torch.tensor(transcript_indices, dtype=torch.long)

# Load the dataset (corrected paths)
video_dir = os.path.join(path, 'data', 's1')  # Videos in 'data/s1'
transcript_dir = os.path.join(path, 'data', 'alignments', 's1')  # Transcripts in 'data/alignments/s1'
dataset = LipReadingDataset(video_dir, transcript_dir)  # Load the entire dataset
dataloader = DataLoader(dataset, batch_size=4, shuffle=True)
print("Dataset loaded. Number of samples:", len(dataset))

Dataset loaded. Number of samples: 1000


### Model Architecture
Define the model architecture for audio-visual fusion and speech recognition. This will typically involve separate branches for processing video (lip movements) and audio, followed by a fusion layer and a final prediction layer.

In [ ]:
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models

class LipReadingModel(nn.Module):
    def __init__(self, num_classes):
        super(LipReadingModel, self).__init__()
        # Video branch (example: using a pre-trained ResNet or a custom CNN)
        # This is a simplified example; a real lip-reading model would be more complex
        self.video_cnn = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(64, 128, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Flatten(),
            nn.Linear(128 * 8 * 8, 512),  # Corrected input size to linear layer
            nn.ReLU(),
            nn.Dropout(0.5)
        )

        # Audio branch (example: using a simple CNN or LSTM on audio features)
        # For this example, we'll use a placeholder as we are focusing on VSR initially
        self.audio_branch = nn.Identity() # Placeholder

        # Fusion layer
        # Assuming both branches output a vector of size 512
        self.fusion_layer = nn.Linear(512 + 512, 512) # Example fusion

        # Transformer for sequence processing
        self.transformer_layer = nn.TransformerEncoderLayer(d_model=512, nhead=8)
        self.transformer_encoder = nn.TransformerEncoder(self.transformer_layer, num_layers=2)


        # Output layer
        self.fc = nn.Linear(512, num_classes)


    def forward(self, video_input, audio_input=None):
        batch_size, time_steps, C, H, W = video_input.size()
        # Reshape to treat time steps as batch size and add a channel dimension
        video_input = video_input.view(batch_size * time_steps, 1, H, W)
        video_features = self.video_cnn(video_input)
        video_features = video_features.view(batch_size, time_steps, 512) # Reshape back to include time steps, explicitly set last dim to 512

        # Placeholder for audio processing
        # audio_features = self.audio_branch(audio_input) # Process audio

        # Simple concatenation for fusion (assuming audio_input is not None and processed)
        # fused_features = torch.cat((video_features, audio_features), dim=-1) # Concatenate

        # For now, just use video features as audio branch is a placeholder
        fused_features = video_features

        # Pass through transformer
        transformer_output = self.transformer_encoder(fused_features)


        # Apply the final linear layer to the transformer output across the time dimension
        output = self.fc(transformer_output)


        return output

# Instantiate the model
# The number of classes should be the size of your vocabulary + 1 for blank
vocab_size = len("abcdefghijklmnopqrstuvwxyz ") + 1 # Plus 1 for blank in CTC
model = LipReadingModel(vocab_size)
print("Model architecture defined.")

Model architecture defined.


/tmp/ipykernel_943/2135786181.py:36: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  self.transformer_encoder = nn.TransformerEncoder(self.transformer_layer, num_layers=2)


In [ ]:
import torch.nn as nn
import torch.optim as optim

# Define the model (CNN + RNN)
class LipReadingModel(nn.Module):
    def __init__(self, vocab_size=28):  # Letters + space + blank for CTC
        super(LipReadingModel, self).__init__()
        self.cnn = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Flatten()
        )
        # Calculate the flattened size after CNN layers
        flattened_size = 64 * 16 * 16

        self.rnn = nn.LSTM(flattened_size, 128, batch_first=True)
        self.fc = nn.Linear(128, vocab_size) # Output layer for CTC

    def forward(self, x):
        batch_size, seq_len, c, h, w = x.shape
        x = x.view(batch_size * seq_len, c, h, w)
        cnn_out = self.cnn(x)
        cnn_out = cnn_out.view(batch_size, seq_len, -1)
        rnn_out, _ = self.rnn(cnn_out)
        out = self.fc(rnn_out)
        # CTC expects log_softmax output
        out = F.log_softmax(out, dim=-1) # Add log_softmax for CTC
        return out

# Create model and setup
vocab_size = len("abcdefghijklmnopqrstuvwxyz ") + 1 # Letters + space + blank
model = LipReadingModel(vocab_size)
# Use CTCLoss
criterion = nn.CTCLoss(blank=vocab_size - 1, zero_infinity=True) # blank is the index of the blank token
optimizer = optim.Adam(model.parameters(), lr=0.001)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)
print("Model created and moved to device.")

Model created and moved to device.


In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

num_epochs = 5
for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    for frames, targets in dataloader:
        frames, targets = frames.to(device), targets.to(device);
        optimizer.zero_grad()
        outputs = model(frames)
        # Calculate input and target lengths for CTCLoss
        input_lengths = torch.full(size=(frames.size(0),), fill_value=outputs.size(1), dtype=torch.long)
        target_lengths = torch.sum(targets != (vocab_size - 1), dim=1)

        loss = criterion(outputs.permute(1, 0, 2), targets, input_lengths, target_lengths) # Permute outputs for CTC

        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {total_loss / len(dataloader)}")

# Save the trained model
torch.save(model.state_dict(), 'lip_reading_model.pth')
print("Model trained and saved as 'lip_reading_model.pth'.")

Epoch 1, Loss: 2.4719550457000734


KeyboardInterrupt: 

In [ ]:
model.eval()
vocab = "abcdefghijklmnopqrstuvwxyz "

def predict(model, frames):
    with torch.no_grad():
        outputs = model(frames.unsqueeze(0).to(device))
        predictions = torch.argmax(outputs, dim=-1).squeeze().cpu().numpy()
        text = ''.join([vocab[i] for i in predictions if i < len(vocab)])
        return text.strip()

# Test on one sample from dataset
if len(dataset) > 0:
    sample_frames, _ = dataset[0]
    predicted_text = predict(model, sample_frames)
    print(f"Predicted text from lip movements: '{predicted_text}'")
else:
    print("No samples to test.")

In [ ]:
# Load the trained lip-reading model
model = LipReadingModel()
model.load_state_dict(torch.load('lip_reading_model.pth'))
model.to(device)
model.eval()

In [ ]:
def process_audio(audio_file_path):
    result = asr_model.transcribe(audio_file_path)
    text = result["text"].strip()
    confidence = np.mean([seg["no_speech_prob"] for seg in result["segments"]])  # Rough confidence (lower is better)
    confidence = 1 - confidence  # Invert for 0-1 scale
    return text, confidence

def process_video(video_path):
      """Real lip-reading using trained model."""
      try:
              cap = cv2.VideoCapture(video_path)
              frames = []
              max_frames_process_video = 25 # Consistent with dataset loading
              while len(frames) < max_frames_process_video:
                          ret, frame = cap.read()
                          if not ret: break
                          mouth = extract_mouth(frame)
                          frames.append(mouth)
              cap.release()

              if len(frames) == 0:
                  return "", 0.0

              while len(frames) < max_frames_process_video:
                  frames.append(np.zeros((64, 64)))
              frames = np.array(frames[:max_frames_process_video])

              frames_tensor = torch.tensor(frames, dtype=torch.float32).unsqueeze(0).to(device)
              predicted_text = predict(model, frames_tensor)
              confidence = 1  # You can improve this later

              return predicted_text, confidence
      except Exception as e:
          print(f"Error processing video: {e}")
          return "", 0.0


def fuse_and_contextualize(asr_text, asr_conf, vsr_text, vsr_conf, context="general conversation"):
    """Fuse outputs and apply context-awareness with refinement."""
    if asr_conf > 0.5 and vsr_conf > 0.5:  # Both good, fuse
        total_conf = asr_conf + vsr_conf + 1e-6
        weight_asr = asr_conf / total_conf
        weight_vsr = vsr_conf / total_conf
        fused_text = f"{weight_asr:.2f}*{asr_text} + {weight_vsr:.2f}*{vsr_text}"
    elif asr_conf > 0.5:  # Use ASR only
        fused_text = asr_text
    elif vsr_conf > 0.5:  # Use VSR only (lip sync)
        fused_text = vsr_text
    else:
        return "Unable to generate captions: Low confidence in both audio and video."

    # Refine: Remove duplicates and make coherent (e.g., for introductions)
    words = fused_text.split()
    refined_words = []
    seen = set()
    for word in words:
        if word.lower() not in seen and not word.startswith("mock"):  # Skip mock artifacts
            refined_words.append(word)
            seen.add(word.lower())
    refined_text = " ".join(refined_words[:15])  # Limit to 15 words for brevity

    # Further refine with T5 for context
    input_text = f"refine and correct: {refined_text} in context of {context}"
    inputs = t5_tokenizer(input_text, return_tensors="pt", max_length=512, truncation=True)
    outputs = t5_model.generate(**inputs, max_length=100, num_beams=4, early_stopping=True)
    final_text = t5_tokenizer.decode(outputs[0], skip_special_tokens=True)
    if not final_text or "refine" in final_text.lower() or len(final_text.split()) < 2:
      final_text = refined_text


    # Context-awareness
    if "meeting" in context.lower() and "write" in final_text.lower():
        final_text = final_text.replace("write", "right")

    return final_text

def generate_captions(audio_path=None, video_path=None, context="general conversation"):
    """Generate captions with logging."""
    asr_text, asr_conf = ("", 0.0) if audio_path is None else process_audio(audio_path)
    vsr_text, vsr_conf = ("", 0.0) if video_path is None else process_video(video_path)

    # Add these debug prints
    print(f"ASR Conf: {asr_conf}, VSR Conf: {vsr_conf}")

    final_captions = fuse_and_contextualize(asr_text, asr_conf, vsr_text, vsr_conf, context)
    print(f"Final Captions: '{final_captions}'")
    return final_captions

In [ ]:
def ui_generate_captions(audio_file, video_file, context):
  captions = generate_captions(audio_file, video_file, context)
  return captions

import gradio as gr

# Assuming your functions like generate_captions are already defined

def ui_generate_captions(audio_file, video_file, context):
    captions = generate_captions(audio_file, video_file, context)
    return captions

with gr.Blocks(
    css="""
        /* Overall theme */
        body {
            background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
            font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;
            color: #333;
            margin: 0;
            padding: 0;
            min-height: 100vh; /* Ensure gradient covers full height */
        }

        /* Container for inputs/outputs */
        .gradio-container {
            max-width: 900px;
            margin: 40px auto; /* Increased top/bottom margin */
            background: rgba(255, 255, 255, 0.98); /* Slightly more opaque */
            border-radius: 20px; /* More rounded corners */
            box-shadow: 0 15px 40px rgba(0,0,0,0.25); /* Stronger shadow */
            padding: 40px; /* Increased padding */
            backdrop-filter: blur(8px); /* Slightly less blur */
        }

        /* Header */
        .gr-markdown h1 {
            color: #4a148c; /* Darker purple */
            text-align: center;
            font-size: 3em; /* Larger font size */
            font-weight: 700; /* Bold font */
            text-shadow: 2px 2px 6px rgba(0,0,0,0.2);
            margin-bottom: 15px; /* Increased margin */
        }
        .gr-markdown p {
            color: #6a1b9a; /* Medium purple */
            text-align: center;
            font-size: 1.3em; /* Larger font size */
            margin-bottom: 30px; /* Increased margin */
        }

        /* Input sections */
        .gr-row {
            display: flex;
            gap: 30px; /* Increased gap */
            margin-bottom: 30px; /* Increased margin */
            flex-wrap: wrap; /* Allow wrapping on smaller screens */
        }
        .gr-audio, .gr-video, .gr-textbox { /* Apply styles to textbox too */
            flex: 1; /* Allow items to grow */
            min-width: 280px; /* Minimum width before wrapping */
            border: 2px dashed #ab47bc; /* Lighter purple border */
            border-radius: 12px; /* More rounded corners */
            padding: 25px; /* Increased padding */
            text-align: center;
            background: #f3e5f5; /* Very light purple */
            transition: all 0.3s ease-in-out; /* Smoother transition */
            box-shadow: 0 2px 10px rgba(0,0,0,0.1); /* Subtle shadow */
        }
        .gr-audio:hover, .gr-video:hover, .gr-textbox:hover {
            border-color: #8e24aa; /* Darker purple on hover */
            background: #e1bee7; /* Light purple on hover */
            transform: translateY(-5px); /* Lift effect on hover */
            box-shadow: 0 8px 20px rgba(0,0,0,0.2); /* Stronger shadow on hover */
        }

        /* Textbox specifically */
        .gr-textbox textarea {
            border: 1px solid #ce93d8; /* Light purple border */
            border-radius: 8px;
            padding: 12px; /* Adjusted padding */
            font-size: 1em;
            line-height: 1.6;
            background: #f8f9fa;
            resize: vertical; /* Allow vertical resizing */
            min-height: 100px; /* Minimum height */
        }
         .gr-textbox label { /* Style for textbox label */
            font-weight: bold;
            color: #4a148c;
            margin-bottom: 8px;
            display: block; /* Make label a block element */
        }


        /* Button */
        .gr-button {
            background: linear-gradient(45deg, #ab47bc, #8e24aa); /* Purple gradient */
            color: white;
            border: none;
            border-radius: 30px; /* More rounded button */
            padding: 15px 40px; /* Increased padding */
            font-size: 1.2em; /* Larger font */
            font-weight: bold;
            cursor: pointer;
            transition: all 0.3s ease;
            box-shadow: 0 5px 15px rgba(0,0,0,0.2);
            margin-top: 20px; /* Added margin above button */
        }
        .gr-button:hover {
            background: linear-gradient(45deg, #8e24aa, #ab47bc); /* Reverse gradient on hover */
            transform: translateY(-3px); /* More pronounced lift */
            box-shadow: 0 8px 25px rgba(0,0,0,0.3);
        }

        /* Output textbox */
        .gr-textbox textarea[readonly] { /* Style for the output textbox */
            background: #e0f2f7; /* Light blue background */
            border: 1px solid #b2ebf2; /* Light blue border */
            color: #004d40; /* Dark green text */
            font-weight: 500;
        }

        /* Footer or additional text */
        .gr-markdown {
            text-align: center;
            color: #4a148c; /* Darker purple for footer text */
            font-size: 1em;
            margin-top: 30px; /* Increased margin */
        }

        /* Mobile responsiveness */
        @media (max-width: 768px) {
            .gr-row {
                flex-direction: column;
                gap: 20px; /* Adjusted gap for smaller screens */
            }
            .gr-markdown h1 {
                font-size: 2.5em; /* Adjusted font size */
            }
            .gradio-container {
                padding: 25px; /* Adjusted padding */
                margin: 20px auto;
            }
             .gr-button {
                width: 100%; /* Full width button on mobile */
                padding: 15px 20px;
             }
        }
    """,
    theme=gr.themes.Soft()  # Optional: Use a built-in theme for extra polish
) as demo:
    gr.Markdown("# 🎥 AV-Captioner: Audio-Visual Speech Recognition")
    gr.Markdown("Upload audio (.wav) and/or video (.mp4) files. Generate captions from lip movements in real-time, even without audio. Perfect for accessibility and noisy environments.")

    with gr.Row():
        audio_input = gr.Audio(label="🎤 Upload Audio File (optional)", type="filepath")
        video_input = gr.Video(label="📹 Upload Video File (optional)")

    context_input = gr.Textbox(label="📝 Context Hint (e.g., 'business meeting')", placeholder="general conversation")

    generate_btn = gr.Button("🚀 Generate Captions")

    output = gr.Textbox(label="📄 Generated Captions", lines=5, placeholder="Captions will appear here...")

    generate_btn.click(ui_generate_captions, inputs=[audio_input, video_input, context_input], outputs=output)

demo.launch(share=True)

'''with gr.Blocks(css="""
     body { background-color: #F5F5F5; font-family: Arial, sans-serif; color: #333; }
    .gr-markdown h1 { color: #007BFF; text-align: center; }
    .gr-button { background-color: #007BFF; color: white; border-radius: 5px; padding: 10px; }
    .gr-textbox { border: 1px solid #007BFF; border-radius: 5px; }
    .gr-audio, .gr-video { border: 2px dashed #007BFF; padding: 20px; text-align: center; }
    """) as demo:
    gr.Markdown("# AV-Captioner: Audio-Visual Speech Recognition")
    gr.Markdown("Upload audio (.wav) and video (.mp4) files to generate captions in noisy or silent environments.")

    with gr.Row():
      audio_input = gr.Audio(label="Upload Audio File", type="filepath")
      video_input = gr.Video(label="Upload Video File")

      context_input = gr.Textbox(label="Context Hint (e.g., 'business meeting')", placeholder="general conversation")
      generate_btn = gr.Button("Generate Captions")
      output = gr.Textbox(label="Generated Captions", lines=5)

      generate_btn.click(ui_generate_captions, inputs=[audio_input, video_input, context_input], outputs=output)

    demo.launch(share=True)'''